# Audio Segment Extraction

> Extract temporal segments from audio files via ffmpeg stream-copy.

In [ ]:
#| default_exp utils.segments

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path
import subprocess
from cjm_capability_ffmpeg.utils.progress import run_ffmpeg_with_progress

def extract_audio_segment(input_path: Path,  # Path to the input audio file
                          output_path: Path,  # Path where the extracted segment is saved
                          start_time: str,  # Start time as "HH:MM:SS" or seconds
                          duration: str,  # Duration as "HH:MM:SS" or seconds
                          verbose: bool = False,  # If True, shows verbose ffmpeg output
                          pbar: bool = False,  # If True, shows a progress bar
                          copy_codec: bool = True,  # Stream-copy without re-encoding (fast)
                        ) -> None:  # Raises subprocess.CalledProcessError if extraction fails
    """Extract a temporal segment from an audio file."""
    # Compute the expected segment duration for the progress bar.
    try:
        segment_duration = float(duration)
    except ValueError:
        time_parts = duration.split(':')
        if len(time_parts) == 3:
            hours = float(time_parts[0])
            minutes = float(time_parts[1])
            seconds = float(time_parts[2])
            segment_duration = hours * 3600 + minutes * 60 + seconds
        else:
            segment_duration = None

    # Place -ss before -i for fast input seeking (seeks to the nearest keyframe).
    cmd = [
        'ffmpeg',
        '-ss', start_time,
        '-i', str(input_path),
        '-t', duration,
    ]
    if copy_codec:
        cmd.extend(['-acodec', 'copy'])
    cmd.extend(['-progress', 'pipe:2', '-y', str(output_path)])

    if pbar:
        run_ffmpeg_with_progress(
            cmd=cmd, total_duration=segment_duration,
            description="Extracting segment", verbose=verbose,
        )
    else:
        result = subprocess.run(
            cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
        )
        if result.returncode != 0:
            raise subprocess.CalledProcessError(result.returncode, cmd)

In [ ]:
assert callable(extract_audio_segment)
print("extract_audio_segment importable")